# AXI test 8: keep or drop the interior tokens in the KV cache? (Qwen2.5-7B-Instruct, free Colab T4)
Cutting the tokens of a deleted service out of the model's memory (its KV cache) saves reading time, but the tokens behind the cut were computed while the deleted service was still visible. Does the model write the same reply as when everything is read again from nothing? This notebook measures it on a real model: **exactness, tokens read and kept, and seconds per step**, for five ways of holding the cache (`fresh`, `exact`, `drop_compact`, `drop_gap`, `keep`).

**The predictions and the reading rules were written and locked BEFORE this run** in `docs/KV_DROP_TEST.md` (15 lines, each with my probability). Nothing in this notebook changes them. A line that comes out NO is a finding, not an error.

**What to know**
* **Cost: $0.** A local open model, no key, no paid service. It needs the **free T4 GPU**: Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save.
* **Time:** about 1 hour in all for Runs A, C and B (an estimate; the quick check in Cell 7 tells you more). Run B is optional.
* **If Colab disconnects:** reconnect, run Cells 2 to 6 again, then run the run-cell that stopped again. Each finished episode is saved to your Google Drive, so it continues where it stopped.
* **Cell 6 is a safety check on the real model.** If it says STOP, do not run the cells after it: send the printout.
* **Do not paste any key or password anywhere.** This notebook needs none.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-kv.zip`

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the 4-bit tools (about 1 minute)

In [ ]:
!pip -q install bitsandbytes accelerate

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and results are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_kv_drop")

### Cell 5: load the model (downloads about 15 GB once; about 5 minutes)

In [ ]:
import json, time, torch
from google.colab import files
from axi.experiments.kv_drop import (KV_MODEL, KV_SEED, KV_LONG_SEED, KV_A_EPISODES, KV_A_STEPS, KV_C_EPISODES, KV_C_STEPS, KV_B_EPISODES, KV_B_STEPS,
                                     run_kv, preflight, verdict_kv, print_kv, print_verdict_kv)
from axi.experiments.kv_torch import build_kv, make_hf_kit

engine, tok, table, model = build_kv(KV_MODEL, True, "cuda")
kit_plain = make_hf_kit(tok, hint_rules=False)      # the system prompt of every earlier 7B test (Run A)
kit_rule = make_hf_kit(tok, hint_rules=True)        # the same prompt with the rule stated (Runs C and B)
T0 = time.time()
def prog(i, n): print(f"  episode {i}/{n}   {time.time() - T0:6.0f} s", end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
res_a = res_c = res_b = None
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1), "| model:", KV_MODEL)

### Cell 6: SAFETY CHECK on the real model (about 1 minute). It prints one line per check and stops if any fails. If it says STOP, do not run the next cells: send me the printout

In [ ]:
checks = preflight(engine, kit_plain, KV_SEED)
for name, ok, detail in checks: print(("ok    " if ok else "FAIL  ") + name + "   [" + detail + "]")
json.dump([dict(name=n, ok=bool(o), detail=d) for n, o, d in checks], open("/content/out/axi_kv_preflight.json", "w"), indent=1); store.persist("axi_kv_preflight.json")
assert all(ok for _, ok, _ in checks), "STOP: a check failed. Do not run the next cells. Send the printout above."
print("\nALL CHECKS OK. Go on to Cell 7.")

### Cell 7: quick check, 1 episode of 4 steps with all five ways (about 2 minutes). If it fails, stop and send me the red error text

In [ ]:
smoke = run_kv(engine, kit_plain, table, 1, 4, 99, hint_rules=False, path="/content/smoke_kv.jsonl", meta=dict(model=KV_MODEL, smoke=True))
print_kv(smoke["summary"], "smoke, 1 episode x 4 steps")

### Cell 8: RUN A, no rule. 8 episodes x 30 steps, all five ways (about 20 minutes; the counter shows episode i/8)

In [ ]:
T0 = time.time(); name = "axi_kv_a"
print("progress file:", store.restore(name + ".jsonl"))
res_a = run_kv(engine, kit_plain, table, KV_A_EPISODES, KV_A_STEPS, KV_SEED, hint_rules=False, path=store.path(name + ".jsonl"), meta=dict(model=KV_MODEL), progress=store.progress(name + ".jsonl", prog))
print(); print_kv(res_a["summary"], "RUN A: no rule")
print_verdict_kv(verdict_kv(res_a["summary"]), "RUN A (lines 14 and 15 need Runs C and B)")
files.download(save(res_a, store.path(name + ".json"))); store.persist(name + ".json")
torch.cuda.empty_cache()

### Cell 9: RUN C, the rule stated in the prompt. The same 8 request scripts (about 20 minutes)

In [ ]:
assert res_a, "Run Cell 8 first."
T0 = time.time(); name = "axi_kv_c"
print("progress file:", store.restore(name + ".jsonl"))
res_c = run_kv(engine, kit_rule, table, KV_C_EPISODES, KV_C_STEPS, KV_SEED, hint_rules=True, path=store.path(name + ".jsonl"), meta=dict(model=KV_MODEL), progress=store.progress(name + ".jsonl", prog))
print(); print_kv(res_c["summary"], "RUN C: rule stated")
print_verdict_kv(verdict_kv(res_a["summary"], res_c["summary"]), "RUN A and RUN C (line 15 needs Run B)")
files.download(save(res_c, store.path(name + ".json"))); store.persist(name + ".json")
torch.cuda.empty_cache()

### Cell 10 (optional): RUN B, long episodes. 2 episodes x 100 steps, rule stated, a different seed (about 20 minutes; the chat arm stops before its memory passes 20,000 tokens)

In [ ]:
assert res_a and res_c, "Run Cells 8 and 9 first."
T0 = time.time(); name = "axi_kv_b"
print("progress file:", store.restore(name + ".jsonl"))
res_b = run_kv(engine, kit_rule, table, KV_B_EPISODES, KV_B_STEPS, KV_LONG_SEED, hint_rules=True, path=store.path(name + ".jsonl"), meta=dict(model=KV_MODEL), progress=store.progress(name + ".jsonl", prog))
print(); print_kv(res_b["summary"], "RUN B: 100-step episodes, rule stated")
files.download(save(res_b, store.path(name + ".json"))); store.persist(name + ".json")
torch.cuda.empty_cache()

### Cell 11: the fifteen reading rules, all at once (prints 'not run' for a run you skipped)

In [ ]:
assert res_a, "Run Cell 8 first."
final = verdict_kv(res_a["summary"], res_c["summary"] if res_c else None, res_b["summary"] if res_b else None)
print_verdict_kv(final, "ALL RUNS")
print("peak GPU memory (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 1), "| minutes since the last run started:", round((time.time() - T0) / 60, 1))

### Done
Files downloaded: `axi_kv_a.json`, `axi_kv_c.json` (and `axi_kv_b.json` if you ran Cell 10); the checks are also in `axi_kv_preflight.json` on your Drive (folder `axi_kv_drop`). Upload the JSON files to the chat, with a screenshot of the printed tables and of the fifteen reading-rule lines. Do not paste any keys anywhere.

When the numbers are in, `docs/KV_DROP_TEST.md` gets a "Result" section **below** the locked predictions; nothing above it changes.